In [0]:
dbutils.widgets.removeAll()

In [0]:
import json

In [0]:
clientContainer = "claimsprocessing"
subGroupConfigPath = "/Workspace/Repos/logi@openhealthagents.org/alphaesai/ClaimsProcessing/DimMember/Gold/Schema/dimMember.json"

In [0]:
# 1. Parse the JSON configuration
with open(subGroupConfigPath, "r") as f:
    config_data = json.load(f)

In [0]:
import json

# Define your configuration paths
clientContainer = "claimsprocessing"
subGroupConfigPath = "/Workspace/Repos/logi@openhealthagents.org/alphaesai/ClaimsProcessing/DimMember/Gold/Schema/dimMember.json"
newDestinationTable = "claimsprocessing.gold.debug_gold_dimmember"

# 1. Parse the JSON configuration
with open(subGroupConfigPath, "r") as f:
    config_data = json.load(f)

# Loop and execute the transformation rules
for entity_row in config_data.get("SubLayerProcessing", []):
    
    # 2. Map the silver source table directly to the 'member' view expected by your SQL script
    for source_row in entity_row.get("SourceTables", []):
        entity_name = source_row.get("Entity")  # This will be 'member'
        source_table = source_row.get("SourceTable").replace("#clientCode", clientContainer)
        
        spark.table(source_table).createOrReplaceTempView(entity_name)
        print(f"-> Mapped source table {source_table} to local view '{entity_name}'")

    # 3. Read and execute the Transformation SQL Script
    sql_script_path = entity_row.get("SQLScriptPath")
    print(f"-> Reading transformation SQL from: {sql_script_path}")
    with open(sql_script_path, "r") as sql_file:
        main_sql_query = sql_file.read()

    # Execute and capture data in a DataFrame
    mDF = spark.sql(main_sql_query)
    
    # 4. Save the transformation result into a physical delta staging table
    staging_table_path = "claimsprocessing.gold.debug_temp_sql_script"
    mDF.write.format("delta").mode("overwrite").saveAsTable(staging_table_path)
    print(f"-> STAGING TABLE WRITTEN: {staging_table_path}")

    # 5. Create the BRAND NEW, empty target dimension table with the identical schema
    # (Using limit(0) copies only the column structure without copying any data rows)
    mDF.limit(0).write.format("delta").mode("overwrite").saveAsTable(newDestinationTable)
    print(f"-> NEW TARGET DIMENSION TABLE CREATED (EMPTY): {newDestinationTable}")
    
    print("\nInitialization complete! You are now fully set up to run your merge script manually in the SQL Editor.")